In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL

load_dotenv()
url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    host=os.getenv("PGHOST"),
    port=int(os.getenv("PGPORT")),
    database=os.getenv("PGDATABASE"),
)
engine = create_engine(url)

pd.read_sql("SELECT is_late, COUNT(*) AS n FROM order_delivery GROUP BY is_late", engine)

,is_late,n
0,False,89936
1,True,6534


In [2]:
from pathlib import Path

query = Path("../sql/12_stats_export.sql").read_text()
df = pd.read_sql(query, engine)
print(df.shape)
print(df.isna().sum())
df.head()

(95824, 9)
order_id            0
customer_state      0
is_late             0
days_vs_estimate    0
review_score        0
n_items             0
items_price         0
freight_total       0
n_sellers           0
dtype: int64


,order_id,customer_state,is_late,days_vs_estimate,review_score,n_items,items_price,freight_total,n_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,RJ,False,-9,5,1,58.90,13.29,1
1,00018f77f2f0320c557190d7a144bdd3,SP,False,-3,4,1,239.90,19.93,1
2,000229ec398224ef6ca0657da4fc703e,MG,False,-14,5,1,199.00,17.87,1
3,00024acbcdf0a6daa1e931b038114c75,SP,False,-6,4,1,12.99,12.79,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,SP,False,-16,5,1,199.90,18.14,1


In [3]:
print(df["is_late"].value_counts())

dist = (df.groupby("is_late")["review_score"]
          .value_counts(normalize=True)
          .unstack()
          .round(3))
print(dist)

print(df.groupby("is_late")["review_score"]
        .agg(["count", "mean", "median", "std"]).round(3))

is_late
False    89443
True      6381
Name: count, dtype: int64
review_score      1      2      3      4      5
is_late                                        
False         0.066  0.026  0.081  0.204  0.623
True          0.538  0.087  0.109  0.102  0.165
         count  mean  median    std
is_late                            
False    89443  4.29     5.0  1.149
True      6381  2.27     1.0  1.571


In [4]:
import numpy as np
from scipy.stats import mannwhitneyu

late = df.loc[df["is_late"], "review_score"].to_numpy().astype(int)
ontime = df.loc[~df["is_late"], "review_score"].to_numpy().astype(int)

u, p = mannwhitneyu(ontime, late, alternative="two-sided")
ps_scipy = u / (len(ontime) * len(late))
print("U =", u, "| p =", p, "| prob. of superiority =", ps_scipy)

def prob_superiority(a, b):
    """P(random a > random b), ties counted as half."""
    pa = np.bincount(a, minlength=6)[1:] / len(a)
    pb = np.bincount(b, minlength=6)[1:] / len(b)
    below_b = np.cumsum(pb) - pb
    return np.sum(pa * (below_b + 0.5 * pb))

ps_mine = prob_superiority(ontime, late)
print("my function:", ps_mine, "| match:", np.isclose(ps_scipy, ps_mine))

rng = np.random.default_rng(42)
boot = np.empty(2000)
for i in range(2000):
    a = rng.choice(ontime, size=len(ontime), replace=True)
    b = rng.choice(late, size=len(late), replace=True)
    boot[i] = prob_superiority(a, b)

print("bootstrap 95% CI:", np.percentile(boot, [2.5, 97.5]))

U = 467430237.0 | p = 0.0 | prob. of superiority = 0.8189958487323371
my function: 0.818995848732337 | match: True
bootstrap 95% CI: [0.8131179 0.8248519]


In [5]:
rng = np.random.default_rng(42)
diffs = np.empty(2000)
for i in range(2000):
    a = rng.choice(ontime, size=len(ontime), replace=True)
    b = rng.choice(late, size=len(late), replace=True)
    diffs[i] = a.mean() - b.mean()
print("mean gap:", ontime.mean() - late.mean())
print("bootstrap 95% CI:", np.percentile(diffs, [2.5, 97.5]))

top_states = df["customer_state"].value_counts().head(6).index
rows = []
for s in top_states:
    sub = df[df["customer_state"] == s]
    a = sub.loc[~sub["is_late"], "review_score"].to_numpy().astype(int)
    b = sub.loc[sub["is_late"], "review_score"].to_numpy().astype(int)
    rows.append({
        "state": s, "n_ontime": len(a), "n_late": len(b),
        "late_rate": round(len(b) / len(sub), 3),
        "prob_sup": round(prob_superiority(a, b), 3),
        "mean_gap": round(a.mean() - b.mean(), 2),
    })
pd.DataFrame(rows)

mean gap: 2.0198955371964127
bootstrap 95% CI: [1.98077705 2.05762429]


,state,n_ontime,n_late,late_rate,prob_sup,mean_gap
0,SP,38480,1786,0.044,0.789,1.78
1,RJ,10755,1456,0.119,0.856,2.33
2,MG,10780,505,0.045,0.801,1.92
3,RS,5005,321,0.060,0.831,2.11
4,PR,4706,194,0.040,0.754,1.63
5,SC,3235,284,0.081,0.820,1.94


In [6]:
import statsmodels.formula.api as smf

model = smf.ols(
    "review_score ~ is_late + C(customer_state)"
    " + np.log1p(items_price) + np.log1p(freight_total) + n_items + n_sellers",
    data=df
).fit(cov_type="HC3")

coef = model.params["is_late[T.True]"]
lo, hi = model.conf_int().loc["is_late[T.True]"]
print(f"is_late coefficient: {coef:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
print("R-squared:", round(model.rsquared, 3))

is_late coefficient: -2.015  95% CI [-2.054, -1.975]
R-squared: 0.182


In [7]:
sens = pd.read_sql(Path("../sql/13_review_dedup_sensitivity.sql").read_text(), engine)
print(sens.shape)

for col in ["score_latest", "score_first", "score_lowest"]:
    g = sens.groupby("is_late")[col].mean()
    print(col, "mean gap:", round(g[False] - g[True], 3))

(95824, 5)
score_latest mean gap: 2.02
score_first mean gap: 2.018
score_lowest mean gap: 2.019
